# Calibration — Methodology & Results

**Forensic question:** How well does the detector perform against a known-positive and known-negative synthetic corpus?

**Phase context:** Phase 12 §6c. This notebook renders the output of `uv run forensics calibrate` — the calibration runner splices AI-generated baseline articles into a real author's timeline at a random pivot date (positive trials) and leaves the corpus unchanged (negative trials). For each trial, the full extract + analysis + survey-scoring pipeline runs end-to-end, and we measure whether the detector fires above the `WEAK` threshold.

**Input artifacts:**
- `data/calibration/calibration_<timestamp>.json` (most recent by mtime)
- `data/preregistration/preregistration_lock.json` (methodology reference)

**Reproduction:** `uv run forensics calibrate --positive-trials 5 --negative-trials 5` then re-render with `quarto render notebooks/11_calibration.ipynb`.

In [ ]:
# | echo: false
from __future__ import annotations

import json
from pathlib import Path

import polars as pl

from forensics.config import get_project_root

ROOT = get_project_root()
CAL_DIR = ROOT / "data" / "calibration"


def _latest_calibration_report() -> Path | None:
    if not CAL_DIR.is_dir():
        return None
    candidates = sorted(
        CAL_DIR.glob("calibration_*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None


report_path = _latest_calibration_report()
if report_path is None:
    print(
        "No calibration report found under data/calibration/. "
        "Run `uv run forensics calibrate` first."
    )
    report = None
else:
    report = json.loads(report_path.read_text(encoding="utf-8"))
    print(f"Loaded: {report_path.relative_to(ROOT)}")
    print(f"n_trials={report.get('n_trials')}")

## Aggregate metrics

- **Sensitivity** — fraction of spliced (positive) trials that fired above `WEAK`.
- **Specificity** — fraction of unmodified (negative) trials that did *not* fire.
- **Precision** — fraction of detections that came from positive trials (no false-positive inflation).
- **F1** — harmonic mean of precision and sensitivity.
- **Median date error (days)** — absolute error between the synthetic splice date and the earliest detected change-point, across positive trials where the detector fired.

In [ ]:
if report is None:
    print("(no calibration data — metrics skipped)")
else:
    rows = [
        {"metric": "sensitivity", "value": report.get("sensitivity")},
        {"metric": "specificity", "value": report.get("specificity")},
        {"metric": "precision", "value": report.get("precision")},
        {"metric": "f1_score", "value": report.get("f1_score")},
        {
            "metric": "median_date_error_days",
            "value": report.get("median_date_error_days"),
        },
        {"metric": "n_trials", "value": report.get("n_trials")},
    ]
    metrics_df = pl.from_dicts(rows)
    print(metrics_df)

## Confusion matrix

Tally of positive / negative trials against detector fire / no-fire outcomes.

In [ ]:
if report is None:
    print("(no calibration data — confusion matrix skipped)")
else:
    import plotly.graph_objects as go  # noqa: I001

    trials = report.get("trials") or []
    if not trials:
        print("(report has no trials)")
    else:
        tp = sum(1 for t in trials if t.get("is_positive") and t.get("detected"))
        fn = sum(1 for t in trials if t.get("is_positive") and not t.get("detected"))
        fp = sum(1 for t in trials if not t.get("is_positive") and t.get("detected"))
        tn = sum(1 for t in trials if not t.get("is_positive") and not t.get("detected"))
        matrix = [[tn, fp], [fn, tp]]
        fig = go.Figure(
            data=go.Heatmap(
                z=matrix,
                x=["predicted: no-fire", "predicted: fire"],
                y=["actual: negative", "actual: positive"],
                text=matrix,
                texttemplate="%{text}",
                colorscale="Blues",
            )
        )
        fig.update_layout(title="Calibration confusion matrix")
        fig.show()
        print(f"TP={tp}  TN={tn}  FP={fp}  FN={fn}")

## Date-error distribution

For every positive trial that fired, how far (in days) was the earliest detected change-point from the synthetic splice date? A tight distribution near zero is evidence the detector recovers *when* the shift occurred, not just *that* one occurred.

In [ ]:
if report is None:
    print("(no calibration data — date-error histogram skipped)")
else:
    import plotly.graph_objects as go  # noqa: I001

    trials = report.get("trials") or []
    errors = [t["date_error_days"] for t in trials if t.get("date_error_days") is not None]
    if not errors:
        print("(no date-error data — no positive trials fired)")
    else:
        fig = go.Figure(go.Histogram(x=errors, nbinsx=min(len(errors), 20)))
        fig.update_layout(
            title="Absolute date error (days) across positive trials",
            xaxis_title="Days between splice and detection",
            yaxis_title="Trials",
        )
        fig.show()

## Interpretation & pre-registration lock

Calibration thresholds match production thresholds: a trial is counted as a detection when the survey scorer classifies the spliced corpus at `WEAK` or stronger. This is the same `_FIRED_STRENGTHS` set used by `forensics.calibration.runner`.

The cell below verifies that the analysis thresholds used here still match the pre-registered lock. A `mismatch` or `missing` status means the calibration numbers above should be read as exploratory.

In [ ]:
from forensics.config import get_settings
from forensics.preregistration import verify_preregistration

try:
    verification = verify_preregistration(get_settings())
    print(f"Pre-registration status: {verification.status}")
    print(verification.message)
except Exception as exc:  # pragma: no cover - presentation-only
    print(f"Pre-registration check skipped: {exc}")